In [0]:
# Find the best model version by minimizing MAE across all songs

# Load predictions data (only songs with actual difficulty > 0)
df_preds = spark.sql("""
    SELECT 
        model_version_num,
        song_id,
        actual_difficulty,
        predicted_difficulty,
        ABS(predicted_difficulty - actual_difficulty) as absolute_error
    FROM acubed.ffr.gold__predictions
    WHERE actual_difficulty > 0
""")

# Calculate MAE for each model version
df_mae_by_version = spark.sql("""
    SELECT 
        model_version_num,
        COUNT(*) as num_predictions,
        AVG(ABS(predicted_difficulty - actual_difficulty)) as mae,
        SQRT(AVG(POW(predicted_difficulty - actual_difficulty, 2))) as rmse,
        AVG(predicted_difficulty - actual_difficulty) as mean_error
    FROM acubed.ffr.gold__predictions
    WHERE actual_difficulty > 0
    GROUP BY model_version_num
    ORDER BY mae ASC
""")

print("="*70)
print("MODEL VERSION PERFORMANCE COMPARISON")
print("="*70)
print("\nMetrics sorted by MAE (Mean Absolute Error):\n")

display(df_mae_by_version)

# Get the best version
best_version = df_mae_by_version.first()

print("\n" + "="*70)
print("BEST MODEL VERSION")
print("="*70)
print(f"\n🏆 Model Version: {best_version['model_version_num']}")
print(f"   MAE:  {best_version['mae']:.2f} difficulty units")
print(f"   RMSE: {best_version['rmse']:.2f} difficulty units")
print(f"   Mean Error: {best_version['mean_error']:.2f}")
print(f"   Total Predictions: {best_version['num_predictions']}")
print("\n" + "="*70)

In [0]:
# Query predictions for the best model version
df_best_model_preds = spark.sql(f"""
    SELECT * 
    FROM acubed.ffr.gold__predictions 
    WHERE model_version_num = {best_version['model_version_num']}
    ORDER BY song_id
""")

display(df_best_model_preds)

In [0]:
import mlflow
from mlflow import MlflowClient

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# Model name in Unity Catalog
model_name = "acubed.ffr.ffr_difficulty_model"
best_model_version = best_version['model_version_num']

# First, remove 'optimal' tag from all versions
print("Removing 'optimal' tag from previous versions...")
try:
    all_versions = client.search_model_versions(f"name='{model_name}'")
    for version in all_versions:
        try:
            # Try to delete the tag - will silently fail if tag doesn't exist
            client.delete_model_version_tag(
                name=model_name,
                version=version.version,
                key="status"
            )
            # print(f"  Removed 'optimal' tag from version {version.version}")
        except:
            # Tag didn't exist on this version, skip
            pass
except Exception as e:
    print(f"  Note: {str(e)}")

# Tag the best model version as "optimal"
mlflow.set_model_version_tag(
    name=model_name,
    version=str(best_model_version),
    key="status",
    value="optimal"
)

# Set the "dev" alias for the best model version (automatically reassigns)
client.set_registered_model_alias(
    name=model_name,
    alias="dev",
    version=str(best_model_version)
)

print(f"\n✅ Successfully tagged model version {best_model_version} as 'optimal'")
print(f"✅ Set alias 'dev' to model version {best_model_version}")
print(f"\n   Model: {model_name}")
print(f"   MAE: {best_version['mae']:.2f} difficulty units")
print(f"\n   Access via: models:/{model_name}@dev")

In [0]:
import mlflow
from mlflow import MlflowClient
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Set up MLflow client
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

model_name = "acubed.ffr.ffr_difficulty_model"

# Get all model versions
all_versions = client.search_model_versions(f"name='{model_name}'")

print(f"Found {len(all_versions)} model versions.\n")

# First, discover what metrics are available by checking the first few versions
print("Discovering available metrics...")
available_metrics = set()
for mv in all_versions[:5]:  # Check first 5 versions
    try:
        run = client.get_run(mv.run_id)
        metrics = run.data.metrics.keys()
        available_metrics.update(metrics)
        print(f"  Version {mv.version}: {', '.join(sorted(metrics))}")
    except:
        pass

print(f"\nAll available metrics: {', '.join(sorted(available_metrics))}\n")

# Try to identify training and validation loss metric names
train_metric = None
val_metric = None

for metric in available_metrics:
    metric_lower = metric.lower()
    if 'train' in metric_lower and 'loss' in metric_lower:
        train_metric = metric
    elif 'val' in metric_lower and 'loss' in metric_lower:
        val_metric = metric
    elif metric_lower in ['training_loss', 'train_mse', 'train_mae']:
        train_metric = metric
    elif metric_lower in ['validation_loss', 'val_mse', 'val_mae']:
        val_metric = metric

if not train_metric or not val_metric:
    print("⚠️  Could not automatically identify training/validation loss metrics.")
    print("\nPlease specify the metric names. Common patterns:")
    print("  - training_loss / validation_loss")
    print("  - train_mse / val_mse")
    print("  - train_mae / val_mae")
    print(f"\nAvailable metrics: {', '.join(sorted(available_metrics))}")
else:
    print(f"Using metrics: train='{train_metric}', val='{val_metric}'")
    print(f"\nFetching metric history for all {len(all_versions)} versions...\n")
    
    # Create single figure
    fig, ax = plt.subplots(1, 1, figsize=(14, 7))
    
    # Collect all training and validation data organized by step
    train_data_by_step = {}
    val_data_by_step = {}
    version_data = []
    
    for mv in all_versions:
        version_num = mv.version
        run_id = mv.run_id
        
        try:
            train_loss = client.get_metric_history(run_id, train_metric)
            val_loss = client.get_metric_history(run_id, val_metric)
            
            if train_loss and val_loss:
                # Collect training data by step
                for m in train_loss:
                    if m.step not in train_data_by_step:
                        train_data_by_step[m.step] = []
                    train_data_by_step[m.step].append(m.value)
                
                # Collect validation data by step
                for m in val_loss:
                    if m.step not in val_data_by_step:
                        val_data_by_step[m.step] = []
                    val_data_by_step[m.step].append(m.value)
                
                train_values = [m.value for m in train_loss]
                val_values = [m.value for m in val_loss]
                
                version_data.append({
                    'version': version_num,
                    'final_train_loss': train_values[-1] if train_values else None,
                    'final_val_loss': val_values[-1] if val_values else None,
                    'epochs': len(train_values)
                })
        except Exception as e:
            pass
    
    # Calculate mean and std for training loss
    train_steps = sorted(train_data_by_step.keys())
    train_means = [np.mean(train_data_by_step[step]) for step in train_steps]
    train_stds = [np.std(train_data_by_step[step]) for step in train_steps]
    
    # Calculate mean and std for validation loss
    val_steps = sorted(val_data_by_step.keys())
    val_means = [np.mean(val_data_by_step[step]) for step in val_steps]
    val_stds = [np.std(val_data_by_step[step]) for step in val_steps]
    
    # Plot mean lines with confidence bands
    ax.plot(train_steps, train_means, linewidth=1, color='#04a3c4', alpha=0.7, label='Training Loss')
    ax.fill_between(train_steps, 
                     [m - s for m, s in zip(train_means, train_stds)],
                     [m + s for m, s in zip(train_means, train_stds)],
                     alpha=0.2, color='#04a3c4')
    
    ax.plot(val_steps, val_means, linewidth=1, color='#FF6B35', alpha=0.2, label='Validation Loss')
    ax.fill_between(val_steps,
                     [m - s for m, s in zip(val_means, val_stds)],
                     [m + s for m, s in zip(val_means, val_stds)],
                     alpha=0.2, color='#FF6B35')
    
    # Add legend
    ax.legend(loc='upper right', fontsize=11)
    
    # Styling
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Mean Average Error (difficulty units)', fontsize=12)
    ax.set_title(f'FFR Difficulty Model Training and Validation Loss', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary
    if version_data:
        df_summary = pd.DataFrame(version_data).sort_values('final_val_loss')
        print(f"\n{'='*70}")
        print("TRAINING SUMMARY - FINAL LOSS VALUES")
        print(f"{'='*70}\n")
        display(df_summary.head(10))
        
        best = df_summary.iloc[0]
        print(f"\n🏆 Best Final Validation Loss: Version {best['version']} with {best['final_val_loss']:.4f}")
    else:
        print("\n⚠️  No training metrics found in MLflow runs.")